In [1]:
### follow this notebook fo rmultiple charge trap implementation

In [6]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
from Minimal_Model_2D import *
import time

np.set_printoptions(threshold=sys.maxsize)

In [7]:
class Minimal_Model_multiple_charge_trap_potential(twoD_Minimal_Model):

    ### overriding a H same spin method to include multiple charge trap 

    def H_same_spin(self, x_ct= np.empty(5), y_ct=np.empty(5), z_ct=np.empty(5), z_im=np.empty(5), z_qw=0, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0):

        #print('a- same spin class overridden')

        M = (hbm/self.meff_sm* self.twodOP.KxSq(kx) + hbm/self.meff_sm* self.twodOP.KySq(ky) - self.chem_pot() + 1/4/hbm*self.meff_sm*self.omega**2*self.twodOP.ysq()  +self.dis_pot_matrix(dis_mag) +self.Disorder_full(uncorr_var, lmbda, seed) )

        for i in range(x_ct.shape[0]):

            M = M + self.charge_pot_lattice(x_ct[i], y_ct[i], z_ct[i], z_im[i], z_qw)
 
        return M


In [8]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.05    # proximitized Ge
MEFF_SM = 0.05     # Ge
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

MY= 5 

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print('mu = ', MU_SM)


UNCORR_VAR = 0  ### uncorrelated variance of disorder
LMBDA = 0
SEED = 58  ## seed for the disorder

# Define the range and the size of the array
x_low = (NSC+1)*AX  # Start of the range (inclusive)
x_high = (NSC+NSM+1)*AX  # End of the range (exclusive)

y_low = 0.0  # Start of the range (inclusive)
y_high = M*AY  # End of the range (exclusive)


z_low = 50.0  # Start of the range (inclusive)
z_high = 51.5  # End of the range (exclusive)

size = 5  # Size of the array

# Generate the array
X_CT = np.random.uniform(x_low, x_high, size)
Y_CT = np.random.uniform(y_low, y_high, size)
Z_CT = np.random.uniform(z_low, z_high, size)
Z_dis = np.ones(size)*(z_high+0.1) - Z_CT
Z_IM  = np.ones(size)*(z_high+0.1) + Z_dis

print(X_CT)
print(Y_CT)
print(Z_CT)
print(Z_IM)


ModCT = Minimal_Model_multiple_charge_trap_potential(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

lengths= 300 100
subband energies= 0.30505912614720065 0.915177378441602
4t= 122.02365045888027
mu =  0.3660709513766408
[4066.97030187 4137.24738801 4278.88766727 4227.68030378 4176.25742877]
[269.35408042 219.32716758 217.28814374 279.97488735 185.19137943]
[51.49498758 50.48867851 50.31744484 50.25951789 51.10501292]
[51.70501242 52.71132149 52.88255516 52.94048211 52.09498708]


In [9]:

H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

Es = np.sort(E)/DELTA

#Ep[i,:] = Es

print(Es)

#print(H.shape)

[-0.58303627 -0.55299467 -0.47208607 -0.29243841  0.29243841  0.47208607
  0.55299467  0.58303627]


In [10]:
ITER = 50

Ep = np.empty((ITER,K_NUM))

for i in range(50):
        
    H = ModCT.H_sys(PHI, 'cu', X_CT, Y_CT, Z_CT, Z_IM)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)/DELTA

    Ep[i,:] = Es

    if i%5==0:

        print(Es)

    #print(H.shape)

    
np.save("cubic_spin_splitting_{}_charge_traps_{}_iter.npy".format(size,ITER))


[-0.58303627 -0.55299467 -0.47208607 -0.29243841  0.29243841  0.47208607
  0.55299467  0.58303627]
